In [5]:
# ============================================================
# CELLA 1 — Installazione librerie e mount Google Drive
# ============================================================

!pip install pyarrow fastparquet -q

from google.colab import drive
drive.mount('/content/drive')

import os
print("Drive montato:", os.path.exists('/content/drive/MyDrive'))


# ============================================================
# CELLA 2 — Configurazione percorsi
# ============================================================
# MODIFICA SOLO DRIVE_ROOT e YEAR.
#
# STRUTTURA ATTESA SU GOOGLE DRIVE:
#
#   MyDrive/
#     thesis_data/                              <- DRIVE_ROOT
#       hmda/
#         hmda_2024.csv                         <- CSV HMDA (non zippato)
#       freddie/
#         historical_data_2024/                 <- sottocartella anno
#           historical_data_2024Q1.zip          <- zip trimestrali Freddie
#           historical_data_2024Q2.zip
#           ...
#       output/                                 <- creata automaticamente
#
# OUTPUT PRODOTTO:
#   output/matched_{YEAR}.csv   <- file CSV con tutte le variabili
#
# SCHEMA COLONNE NEL CSV OUTPUT:
#
#   [A] IDENTIFICATORI
#       loan_sequence_number        ID univoco Freddie Mac
#       match_year                  anno di elaborazione
#
#   [B] TUTTE LE 32 VARIABILI FREDDIE MAC (nomi originali)
#       credit_score, first_payment_date, first_time_homebuyer,
#       maturity_date, msa_freddie, mi_pct, num_units_freddie,
#       occupancy_status (P/S/I), original_cltv, original_dti,
#       original_upb, original_ltv, original_interest_rate,
#       channel, ppm_flag, amortization_type, property_state,
#       property_type, postal_code, loan_purpose_freddie (P/C/N/R),
#       original_loan_term, num_borrowers, seller_name, servicer_name,
#       super_conforming_flag, pre_relief_refi_seq, special_eligibility,
#       relief_refi_indicator, property_valuation, io_indicator,
#       mi_cancellation
#
#   [C] VARIABILI DI MATCH — valori HMDA originali (codici numerici)
#       state_code          stato (es. "CA")
#       msa                 MSA/MD (codice 5 cifre)
#       loan_amount         importo esatto HMDA (non arrotondato)
#       loan_amount_r       importo arrotondato al $1,000 (chiave di match)
#       interest_rate       tasso interesse
#       loan_term           durata mutuo in mesi
#       num_units           numero unita' abitative (1/2/3/4)
#       occupancy_type      codice HMDA occupancy (1/2/3)
#       loan_purpose        codice HMDA loan purpose (1/31/32)
#
#   [D] VARIABILI DI MATCH — valori Freddie originali (pre-mappatura)
#       occupancy_status_orig   codice Freddie P/S/I (prima della mappatura)
#       loan_purpose_orig       codice Freddie P/C/N/R (prima della mappatura)
#
#   [E] DATI DEMOGRAFICI HMDA
#       derived_race            razza (derivata dal sistema HMDA)
#       applicant_race_1..5     razza dichiarata (fino a 5 voci)
#       co_applicant_race_1..5  razza co-richiedente
#       derived_sex             sesso (derivato)
#       applicant_sex           sesso richiedente (codice)
#       co_applicant_sex        sesso co-richiedente (codice)
#       applicant_age           eta' richiedente (fascia)
#       co_applicant_age        eta' co-richiedente (fascia)
#       applicant_age_above_62  flag over 62 richiedente
#       co_applicant_age_above_62 flag over 62 co-richiedente

import pandas as pd
import numpy as np
import glob
import gc
import zipfile
import shutil

# ---- UNICI DUE VALORI DA MODIFICARE ----
DRIVE_ROOT = '/content/drive/MyDrive/thesis_data'
YEAR       = 2024
# ----------------------------------------

HMDA_DIR      = os.path.join(DRIVE_ROOT, 'hmda')
FREDDIE_DIR   = os.path.join(DRIVE_ROOT, 'freddie')
FREDDIE_LOCAL = '/content/freddie_local'
OUTPUT_DIR    = os.path.join(DRIVE_ROOT, 'output')
os.makedirs(OUTPUT_DIR,    exist_ok=True)
os.makedirs(FREDDIE_LOCAL, exist_ok=True)

CHUNK_SIZE = 200_000

print(f"Anno selezionato : {YEAR}")
print(f"HMDA dir         : {HMDA_DIR}")
print(f"Freddie zip dir  : {FREDDIE_DIR}")
print(f"Freddie locale   : {FREDDIE_LOCAL}")
print(f"Output dir       : {OUTPUT_DIR}")

for label, path in [("HMDA", HMDA_DIR), ("Freddie zip", FREDDIE_DIR)]:
    print(f"  {'OK' if os.path.exists(path) else 'ERRORE'} {label}")


# ============================================================
# CELLA 2b — Estrazione zip Freddie nel disco locale Colab
# ============================================================

def unzip_freddie_year(year: int) -> list:
    """
    Cerca i file zip Freddie su Drive (con o senza sottocartella anno),
    estrae i file di origination nel disco locale Colab.
    Restituisce la lista ordinata dei percorsi .txt estratti.
    """
    zip_pattern_sub  = os.path.join(FREDDIE_DIR, f"historical_data_{year}",
                                    f"historical_data_{year}Q*.zip")
    zip_pattern_flat = os.path.join(FREDDIE_DIR, f"historical_data_{year}Q*.zip")
    zip_files = glob.glob(zip_pattern_sub) or glob.glob(zip_pattern_flat)

    if not zip_files:
        raise FileNotFoundError(
            f"\nNessun file zip trovato per {year}.\n"
            f"Pattern cercati:\n"
            f"  {zip_pattern_sub}\n"
            f"  {zip_pattern_flat}\n"
            f"Struttura attesa: freddie/historical_data_{year}/"
            f"historical_data_{year}Q1.zip ..."
        )

    print(f"Trovati {len(zip_files)} file zip per {year}:")
    extracted_txts = []

    for zpath in sorted(zip_files):
        zname    = os.path.basename(zpath)
        qname    = zname.replace(".zip", "")
        txt_dest = os.path.join(FREDDIE_LOCAL, f"{qname}.txt")

        if os.path.exists(txt_dest):
            size_mb = os.path.getsize(txt_dest) / 1e6
            print(f"  {zname} -> gia' estratto ({size_mb:.0f} MB), skip")
            extracted_txts.append(txt_dest)
            continue

        print(f"  Estraggo {zname} ...", end=" ", flush=True)

        with zipfile.ZipFile(zpath, 'r') as z:
            contents = z.namelist()
            orig_files = [f for f in contents
                          if "time" not in f.lower() and f.endswith(".txt")]
            if not orig_files:
                print(f"\n  ATTENZIONE: nessun file origination in {zname}")
                print(f"  Contenuto: {contents}")
                continue
            with z.open(orig_files[0]) as src, open(txt_dest, 'wb') as dst:
                shutil.copyfileobj(src, dst)

        size_mb = os.path.getsize(txt_dest) / 1e6
        print(f"OK ({size_mb:.0f} MB)")
        extracted_txts.append(txt_dest)

    return sorted(extracted_txts)


print(f"\nEstraggo Freddie {YEAR}...")
freddie_txt_files = unzip_freddie_year(YEAR)
print(f"\nFile disponibili: {len(freddie_txt_files)}")
for f in freddie_txt_files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)")


# ============================================================
# CELLA 3 — Layout colonne e mappature
# ============================================================

# Tutte e 32 le colonne del file origination Freddie Mac (ordine del manuale)
FREDDIE_ORIG_COLS = [
    "credit_score",           # col 1
    "first_payment_date",     # col 2  -> YYYYMM
    "first_time_homebuyer",   # col 3
    "maturity_date",          # col 4  -> YYYYMM
    "msa",                    # col 5  -> codice MSA/MD
    "mi_pct",                 # col 6
    "num_units",              # col 7  -> 1/2/3/4
    "occupancy_status",       # col 8  -> P/S/I
    "original_cltv",          # col 9
    "original_dti",           # col 10
    "original_upb",           # col 11 -> importo al $1,000
    "original_ltv",           # col 12
    "original_interest_rate", # col 13
    "channel",                # col 14
    "ppm_flag",               # col 15
    "amortization_type",      # col 16
    "property_state",         # col 17
    "property_type",          # col 18
    "postal_code",            # col 19 -> ###00
    "loan_sequence_number",   # col 20 -> ID univoco
    "loan_purpose",           # col 21 -> P/C/N/R
    "original_loan_term",     # col 22 -> mesi
    "num_borrowers",          # col 23
    "seller_name",            # col 24
    "servicer_name",          # col 25
    "super_conforming_flag",  # col 26
    "pre_relief_refi_seq",    # col 27
    "special_eligibility",    # col 28
    "relief_refi_indicator",  # col 29
    "property_valuation",     # col 30
    "io_indicator",           # col 31
    "mi_cancellation",        # col 32
]
# NOTA: carichiamo TUTTE le 32 colonne -> usecols=None (nessun filtro)

# Mappatura Freddie -> HMDA: occupancy
FREDDIE_OCC_MAP = {"P": 1, "S": 2, "I": 3}

# Mappatura Freddie -> HMDA: loan purpose
FREDDIE_PURPOSE_MAP = {"P": 1, "C": 32, "N": 31, "R": 31}

# Chiavi di match (colonne con nomi HMDA, dopo rinomina Freddie)
MATCH_KEYS = [
    "state_code",      # property_state rinominato
    "msa",             # msa (stesso nome)
    "loan_amount_r",   # original_upb rinominato
    "interest_rate",   # original_interest_rate rinominato
    "loan_term",       # original_loan_term rinominato
    "num_units",       # num_units (stesso nome)
    "occupancy_type",  # occupancy_status mappato in codice HMDA
    "loan_purpose",    # loan_purpose mappato in codice HMDA
]

# Colonne demografiche HMDA da tenere sempre
HMDA_DEMO_COLS = [
    # Razza
    "derived_race",
    "applicant_race_1", "applicant_race_2", "applicant_race_3",
    "applicant_race_4", "applicant_race_5",
    "co_applicant_race_1", "co_applicant_race_2", "co_applicant_race_3",
    "co_applicant_race_4", "co_applicant_race_5",
    # Sesso
    "derived_sex",
    "applicant_sex",
    "co_applicant_sex",
    # Eta'
    "applicant_age",
    "co_applicant_age",
    "applicant_age_above_62",
    "co_applicant_age_above_62",
]

# Colonne di match HMDA da tenere (valori originali HMDA, pre-rinomina)
HMDA_MATCH_ORIG_COLS = [
    "state_code",       # stato
    "derived_msa_md",   # MSA/MD originale HMDA (prima della rinomina in "msa")
    "loan_amount",      # importo esatto (non arrotondato)
    "loan_amount_r",    # importo arrotondato al $1,000 (chiave di match)
    "interest_rate",    # tasso
    "loan_term",        # durata mesi
    "num_units",        # numero unita' (rinominato da total_units)
    "occupancy_type",   # codice HMDA 1/2/3
    "loan_purpose",     # codice HMDA 1/31/32
]

print("Layout e mappature caricati.")
print(f"  Colonne Freddie da caricare : {len(FREDDIE_ORIG_COLS)}")
print(f"  Colonne demo HMDA           : {len(HMDA_DEMO_COLS)}")
print(f"  Chiavi di match             : {len(MATCH_KEYS)}")


# ============================================================
# CELLA 4 — Caricamento e preparazione Freddie Mac
# ============================================================
# Carica TUTTE e 32 le colonne Freddie.
# Salva anche i valori originali P/S/I e P/C/N/R prima della mappatura.

def load_and_prepare_freddie(txt_files: list, year: int) -> pd.DataFrame:
    """
    Carica tutti i file .txt Freddie (disco locale Colab).
    - Carica tutte e 32 le colonne originali
    - Salva occupancy e loan_purpose PRIMA della mappatura in colonne _orig
    - Applica mappatura -> crea colonne con nomi HMDA per il merge
    - Rinomina le colonne chiave con i nomi HMDA
    """
    if not txt_files:
        raise ValueError("Nessun file .txt da caricare. Riesegui la Cella 2b.")

    frames = []
    for f in txt_files:
        df = pd.read_csv(
            f,
            sep="|",
            header=None,
            names=FREDDIE_ORIG_COLS,   # tutte e 32
            usecols=None,              # carica TUTTO
            dtype=str,
            low_memory=False,
        )
        print(f"  {os.path.basename(f)}: {len(df):,} righe  "
              f"({os.path.getsize(f)/1e6:.0f} MB)")
        frames.append(df)

    freddie = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    print(f"Totale caricato: {len(freddie):,} righe")

    # --- FILTRO ANNO ---
    freddie["fp_year"] = freddie["first_payment_date"].str[:4]
    freddie = freddie[freddie["fp_year"].isin([str(year), str(year - 1)])]
    print(f"Dopo filtro anno: {len(freddie):,} righe")

    # --- SALVA VALORI ORIGINALI FREDDIE (pre-mappatura) ---
    # Queste colonne saranno nel CSV finale come riferimento
    freddie["occupancy_status_orig"] = freddie["occupancy_status"].copy()
    freddie["loan_purpose_orig"]     = freddie["loan_purpose"].copy()

    # --- ZIP3 ---
    freddie["zip3"] = freddie["postal_code"].str[:3]

    # --- MAPPATURA Freddie -> HMDA ---
    freddie["occupancy_type"] = freddie["occupancy_status"].map(FREDDIE_OCC_MAP)
    freddie["loan_purpose"]   = freddie["loan_purpose"].map(FREDDIE_PURPOSE_MAP)

    # --- RINOMINA colonne chiave con nomi HMDA ---
    freddie.rename(columns={
        "property_state":         "state_code",
        "original_upb":           "loan_amount_r",
        "original_interest_rate": "interest_rate",
        "original_loan_term":     "loan_term",
        # "msa" rimane "msa"
        # "num_units" rimane "num_units"
    }, inplace=True)

    # --- CONVERSIONI NUMERICHE ---
    for col in ["loan_amount_r", "interest_rate", "loan_term", "num_units",
                "original_ltv", "original_cltv", "original_dti",
                "credit_score", "mi_pct"]:
        if col in freddie.columns:
            freddie[col] = pd.to_numeric(freddie[col], errors="coerce")

    freddie["msa"]        = freddie["msa"].str.strip().fillna("")
    freddie["state_code"] = freddie["state_code"].str.strip()

    # Rimuovi righe con NaN sulle chiavi di match
    before = len(freddie)
    freddie.dropna(subset=MATCH_KEYS, inplace=True)
    print(f"Dopo dropna chiavi: {len(freddie):,} righe "
          f"(rimossi {before - len(freddie):,})")

    ram_mb = freddie.memory_usage(deep=True).sum() / 1e6
    print(f"RAM Freddie: {ram_mb:.0f} MB")
    return freddie


print(f"\nCarico Freddie Mac {YEAR}...")
freddie_ready = load_and_prepare_freddie(freddie_txt_files, YEAR)
print(f"\nFreddie pronto: {len(freddie_ready):,} righe, "
      f"{len(freddie_ready.columns)} colonne")


# ============================================================
# CELLA 5 — Match con HMDA a chunk
# ============================================================
# Mantiene nel chunk HMDA:
#   - le variabili di match nei valori originali HMDA
#   - le colonne demografiche (race, sex, age)
#   - activity_year e tutte le altre colonne HMDA filtrate utili
#
# NON vengono mantenute le colonne HMDA non rilevanti (es. hoepa_status,
# balloon_payment, ecc.) per tenere il CSV leggibile.
# Se vuoi TUTTO l'HMDA, imposta KEEP_ALL_HMDA = True qui sotto.

KEEP_ALL_HMDA = False   # True = tieni tutte le colonne HMDA nel CSV

# Colonne HMDA da conservare sempre (oltre alle chiavi di match e demografiche)
HMDA_EXTRA_COLS = [
    "activity_year",
    "lei",
    "action_taken",
    "purchaser_type",
    "loan_type",
    "property_type",          # tipo proprieta'
    "lien_status",
    "reverse_mortgage",
    "open_end_line_of_credit",
    "business_or_commercial",
    "conforming_loan_limit",
    "derived_loan_product_type",
    "derived_dwelling_category",
    "county_code",
    "census_tract",
    "applicant_ethnicity_1",
    "co_applicant_ethnicity_1",
    "derived_ethnicity",
    "income",
    "rate_spread",
    "hoepa_status",
    "total_loan_costs",
    "origination_charges",
    "discount_points",
    "lender_credits",
    "loan_to_value_ratio",
    "intro_rate_period",
    "negative_amortization",
    "interest_only_payment",
    "balloon_payment",
    "other_nonamortizing_features",
    "property_value",
    "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest",
    "submission_of_application",
    "initially_payable_to_institution",
    "aus_1",
    "denial_reason_1",
    "tract_population",
    "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income",
    "tract_to_msa_income_percentage",
    "tract_owner_occupied_units",
    "tract_one_to_four_family_homes",
    "tract_median_age_of_housing_units",
]


def find_hmda_file(year: int) -> str:
    candidates = [
        os.path.join(HMDA_DIR, f"hmda_{year}.csv"),
        os.path.join(HMDA_DIR, f"{year}_public_lar.csv"),
        os.path.join(HMDA_DIR, f"hmda_{year}_nationwide_all-records_labels.csv"),
        os.path.join(HMDA_DIR, f"hmda_{year}_nationwide_all-records_codes.csv"),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    found = glob.glob(os.path.join(HMDA_DIR, f"*{year}*.csv"))
    if found:
        return found[0]
    raise FileNotFoundError(
        f"\nNessun file HMDA trovato per {year} in '{HMDA_DIR}'.\n"
        f"Nomi accettati: hmda_{year}.csv, {year}_public_lar.csv, ..."
    )


def prepare_hmda_chunk(chunk: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    Filtra e prepara un chunk HMDA per il merge.
    Conserva: chiavi di match + demografia + colonne extra definite sopra.
    """
    # Normalizza nomi: trattino -> underscore
    chunk.rename(columns=lambda c: c.replace("-", "_"), inplace=True)

    # --- FILTRI ---
    if "activity_year" in chunk.columns:
        chunk = chunk[chunk["activity_year"].astype(str) == str(year)]
    if chunk.empty: return chunk

    chunk = chunk[chunk["purchaser_type"].astype(str) == "3"]
    if chunk.empty: return chunk

    chunk = chunk[chunk["action_taken"].astype(str) == "1"]
    if chunk.empty: return chunk

    chunk = chunk[chunk["total_units"].isin(["1", "2", "3", "4"])]
    if chunk.empty: return chunk

    # Conversioni numeriche sulle chiavi
    for col in ["loan_amount", "interest_rate", "loan_term",
                "total_units", "occupancy_type", "loan_purpose"]:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk[chunk["loan_purpose"].isin([1, 31, 32])]
    if chunk.empty: return chunk

    # Arrotondamento importo
    chunk["loan_amount_r"] = (chunk["loan_amount"] / 1000).round() * 1000

    # Rinomina per allinearsi alle MATCH_KEYS
    rename_map = {}
    if "total_units"    in chunk.columns: rename_map["total_units"]    = "num_units"
    if "derived_msa_md" in chunk.columns: rename_map["derived_msa_md"] = "msa"
    chunk.rename(columns=rename_map, inplace=True)

    if "state_code" in chunk.columns:
        chunk["state_code"] = chunk["state_code"].str.strip()
    if "msa" in chunk.columns:
        chunk["msa"] = chunk["msa"].str.strip().fillna("")

    chunk.dropna(subset=MATCH_KEYS, inplace=True)
    if chunk.empty: return chunk

    # --- SELEZIONE COLONNE ---
    if KEEP_ALL_HMDA:
        # Tieni tutto HMDA
        return chunk

    # Tieni solo le colonne di interesse
    cols_to_keep = (
        MATCH_KEYS +
        ["loan_amount",         # importo originale HMDA (non arrotondato)
         "derived_msa_md" if "derived_msa_md" in chunk.columns else "msa"] +
        [c for c in HMDA_DEMO_COLS  if c in chunk.columns] +
        [c for c in HMDA_EXTRA_COLS if c in chunk.columns]
    )
    # Deduplica mantenendo l'ordine
    seen = set()
    cols_to_keep = [c for c in cols_to_keep
                    if c in chunk.columns and not (c in seen or seen.add(c))]

    return chunk[cols_to_keep]


# --- ESECUZIONE DEL MATCH A CHUNK ---

hmda_path = find_hmda_file(YEAR)
print(f"File HMDA : {hmda_path}")
print(f"Chunk size: {CHUNK_SIZE:,} righe\n")

chunk_results  = []
chunk_num      = 0
total_rows     = 0
total_filtered = 0
total_matched  = 0

reader = pd.read_csv(hmda_path, dtype=str, chunksize=CHUNK_SIZE, low_memory=False)

for chunk in reader:
    chunk_num  += 1
    total_rows += len(chunk)

    chunk_clean = prepare_hmda_chunk(chunk, YEAR)
    del chunk
    gc.collect()

    if chunk_clean.empty:
        if chunk_num % 10 == 0:
            print(f"  Chunk {chunk_num:3d} | lette {total_rows:>10,} | "
                  f"filtrate {total_filtered:>8,} | match {total_matched:>7,}")
        continue

    total_filtered += len(chunk_clean)

    matched = pd.merge(
        chunk_clean, freddie_ready,
        on=MATCH_KEYS,
        how="inner",
        suffixes=("_hmda", "_freddie"),
    )
    del chunk_clean
    gc.collect()

    if not matched.empty:
        total_matched += len(matched)
        chunk_results.append(matched)

    if chunk_num % 10 == 0:
        print(f"  Chunk {chunk_num:3d} | lette {total_rows:>10,} | "
              f"filtrate {total_filtered:>8,} | match {total_matched:>7,}")

print(f"\nCompletato.")
print(f"  Chunk totali     : {chunk_num:,}")
print(f"  Righe HMDA lette : {total_rows:,}")
print(f"  Dopo filtri      : {total_filtered:,}")
print(f"  Match grezzi     : {total_matched:,}")


# ============================================================
# CELLA 6 — Pulizia collisioni, ordinamento colonne, salvataggio CSV
# ============================================================

if not chunk_results:
    print("Nessun match trovato. Controlla percorsi e anno.")
else:
    print("Combino i chunk...")
    all_matches = pd.concat(chunk_results, ignore_index=True)
    del chunk_results
    gc.collect()
    print(f"Match totali prima della pulizia: {len(all_matches):,}")

    # --- RIMOZIONE COLLISIONI ---
    dup_freddie  = all_matches.duplicated(subset=["loan_sequence_number"], keep=False)
    dup_hmda     = all_matches.duplicated(subset=MATCH_KEYS, keep=False)
    is_ambiguous = dup_freddie | dup_hmda
    n_amb        = is_ambiguous.sum()
    n_tot        = len(all_matches)

    print(f"Collisioni rimosse : {n_amb:,}  ({100*n_amb/n_tot:.1f}%)")
    print(f"  di cui Freddie   : {dup_freddie.sum():,}")
    print(f"  di cui HMDA      : {dup_hmda.sum():,}")

    final = all_matches[~is_ambiguous].copy()
    del all_matches
    gc.collect()

    n_clean = len(final)
    print(f"Match puliti 1-a-1 : {n_clean:,}")
    print(f"Match rate Freddie : {n_clean:,} / {len(freddie_ready):,} = "
          f"{100*n_clean/len(freddie_ready):.1f}%")

    final["match_year"] = YEAR

    # -------------------------------------------------------
    # ORDINAMENTO COLONNE NEL CSV FINALE
    # -------------------------------------------------------
    # [A] Identificatori
    id_cols = ["loan_sequence_number", "match_year"]

    # [B] Tutte le variabili Freddie originali (con nomi rinominati)
    # Nota: property_state -> state_code, original_upb -> loan_amount_r, ecc.
    # Usiamo i nomi come esistono nel DataFrame dopo la preparazione
    freddie_cols_in_output = [
        "credit_score", "first_payment_date", "first_time_homebuyer",
        "maturity_date",
        "msa",                  # col 5 freddie (coincide con chiave match)
        "mi_pct",
        "num_units",            # col 7 (coincide con chiave match)
        "occupancy_status_orig", # col 8 originale P/S/I (pre-mappatura)
        "original_cltv",
        "original_dti",
        "loan_amount_r",        # col 11 = original_upb rinominato
        "original_ltv",
        "interest_rate",        # col 13 = original_interest_rate rinominato
        "channel",
        "ppm_flag",
        "amortization_type",
        "state_code",           # col 17 = property_state rinominato
        "property_type",        # col 18 (puo' esistere anche in HMDA -> _freddie)
        "postal_code",
        # loan_sequence_number gia' in [A]
        "loan_purpose_orig",    # col 21 originale P/C/N/R (pre-mappatura)
        "loan_term",            # col 22 = original_loan_term rinominato
        "num_borrowers",
        "seller_name",
        "servicer_name",
        "super_conforming_flag",
        "pre_relief_refi_seq",
        "special_eligibility",
        "relief_refi_indicator",
        "property_valuation",
        "io_indicator",
        "mi_cancellation",
        "fp_year",
        "zip3",
    ]

    # [C] Variabili di match — valori HMDA originali (codici numerici)
    match_cols = [
        "loan_amount",          # importo esatto HMDA (non arrotondato)
        "loan_amount_r",        # importo arrotondato (chiave di match)
        "occupancy_type",       # codice HMDA 1/2/3
        "loan_purpose",         # codice HMDA 1/31/32
        # state_code, msa, interest_rate, loan_term, num_units
        # gia' presenti in [B] perche' condivisi dopo rinomina
    ]

    # [D] Variabili demografiche HMDA
    demo_cols = [c for c in HMDA_DEMO_COLS if c in final.columns]

    # [E] Colonne HMDA extra
    extra_cols = [c for c in HMDA_EXTRA_COLS if c in final.columns]

    # Costruisci ordine finale, deduplicando
    ordered = []
    seen = set()
    for group in [id_cols, freddie_cols_in_output, match_cols, demo_cols, extra_cols]:
        for c in group:
            # Gestisci colonne duplicate da merge (con suffisso _hmda o _freddie)
            if c in final.columns and c not in seen:
                ordered.append(c)
                seen.add(c)
            # Se c'e' la versione con suffisso, aggiungila anche
            for suffix in ["_freddie", "_hmda"]:
                c_suf = f"{c}{suffix}"
                if c_suf in final.columns and c_suf not in seen:
                    ordered.append(c_suf)
                    seen.add(c_suf)

    # Aggiungi eventuali colonne rimaste fuori
    remaining = [c for c in final.columns if c not in seen]
    if remaining:
        print(f"\nColonne extra aggiunte in coda: {remaining}")
    ordered += remaining

    final = final[ordered]

    # --- SALVATAGGIO CSV ---
    out_path = os.path.join(OUTPUT_DIR, f"matched_{YEAR}.csv")
    final.to_csv(out_path, index=False)

    size_mb = os.path.getsize(out_path) / 1e6
    print(f"\nSalvato: {out_path}  ({size_mb:.1f} MB)")
    print(f"Righe  : {len(final):,}")
    print(f"Colonne: {len(final.columns)}")

    # Riepilogo colonne per gruppo
    print("\n--- COLONNE NEL CSV FINALE ---")
    print(f"[A] Identificatori         : {[c for c in id_cols if c in final.columns]}")
    print(f"[B] Freddie Mac (32 col)   : {[c for c in freddie_cols_in_output if c in final.columns]}")
    print(f"[C] Match vars HMDA orig   : {[c for c in match_cols if c in final.columns]}")
    print(f"[D] Demografica HMDA       : {demo_cols}")
    print(f"[E] Extra HMDA             : {[c for c in extra_cols]}")
    if remaining:
        print(f"[F] Altre colonne          : {remaining}")

    display(final.head(3))


# ============================================================
# CELLA 7 — (OPZIONALE) Combina tutti gli anni in un unico CSV
# ============================================================
# Esegui SOLO dopo aver completato tutte le elaborazioni annuali (2017-2024).

files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "matched_20*.csv")))
print(f"File CSV trovati: {len(files)}")
for f in files:
    n = pd.read_csv(f, nrows=0).shape[1]  # conta solo colonne senza caricare tutto
    print(f"  {os.path.basename(f)}")

if files:
    # Legge tutto solo se si vuole davvero combinare
    print("\nCombino tutti gli anni... (operazione pesante)")
    final_all = pd.concat(
        [pd.read_csv(f, dtype=str, low_memory=False) for f in files],
        ignore_index=True
    )
    final_all["match_year"] = final_all["match_year"].astype(int)
    out_final = os.path.join(OUTPUT_DIR, "matched_2017_2024.csv")
    final_all.to_csv(out_final, index=False)
    size_mb = os.path.getsize(out_final) / 1e6
    print(f"\nDataset combinato: {len(final_all):,} righe")
    print(f"Salvato: {out_final}  ({size_mb:.1f} MB)")
    print("\nRighe per anno:")
    print(final_all.groupby("match_year").size().rename("righe").to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montato: True
Anno selezionato : 2024
HMDA dir         : /content/drive/MyDrive/thesis_data/hmda
Freddie zip dir  : /content/drive/MyDrive/thesis_data/freddie
Freddie locale   : /content/freddie_local
Output dir       : /content/drive/MyDrive/thesis_data/output
  OK HMDA
  OK Freddie zip

Estraggo Freddie 2024...
Trovati 4 file zip per 2024:
  historical_data_2024Q1.zip -> gia' estratto (34 MB), skip
  historical_data_2024Q2.zip -> gia' estratto (39 MB), skip
  historical_data_2024Q3.zip -> gia' estratto (46 MB), skip
  historical_data_2024Q4.zip -> gia' estratto (44 MB), skip

File disponibili: 4
  historical_data_2024Q1.txt  (34 MB)
  historical_data_2024Q2.txt  (39 MB)
  historical_data_2024Q3.txt  (46 MB)
  historical_data_2024Q4.txt  (44 MB)
Layout e mappature caricati.
  Colonne Freddie da caricare : 32
  Colonne demo HMDA           : 18
  Chiavi 

,loan_sequence_number,match_year,credit_score,first_payment_date,first_time_homebuyer,maturity_date,msa,mi_pct,num_units,occupancy_status_orig,...,aus_1,denial_reason_1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units,occupancy_status
4,F24Q30248818,2024,733,202409,N,203908,35980,0,1,P,...,2,10,2786,11.63,105900,131.0,1153,1418,37,P
7,F24Q30286494,2024,694,202410,N,205409,26900,0,1,P,...,2,10,7063,7.38,98600,131.0,2056,2517,35,P
11,F24Q30212838,2024,667,202410,Y,205409,47764,18,1,P,...,2,10,2922,91.2,128300,71.0,655,803,75,P


File CSV trovati: 1
  matched_2024.csv

Combino tutti gli anni... (operazione pesante)

Dataset combinato: 36,906 righe
Salvato: /content/drive/MyDrive/thesis_data/output/matched_2017_2024.csv  (17.7 MB)

Righe per anno:
match_year
2024    36906
